In [ ]:
%pip install mediapipe opencv-python

In [ ]:
%pip uninstall mediapipe -y
%pip install mediapipe==0.10.5
%pip install protobuf==3.20.3 --force-reinstall

In [ ]:
import os
import mediapipe as mp
import cv2
import numpy as np
import uuid
import math
import time

In [ ]:
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands


In [ ]:
cap = cv2.VideoCapture(0)
with mp_hands.Hands(min_detection_confidence=0.8, min_tracking_confidence=0.5) as hands:
    while cap.isOpened():
        ret, frame = cap.read()
        frame = cv2.flip(frame, 1)

        #BGR 2 RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        #Set flag
        image.flags.writeable = False

        #Detections
        results = hands.process(image)

        #Set flag to true
        image.flags.writeable = True

        #RGB 2 BGR
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        #Detections
        print(results)
        
        #Rendering results
        if results.multi_hand_landmarks:
            for num, hand in enumerate(results.multi_hand_landmarks):
                mp_drawing.draw_landmarks(image, hand, mp_hands.HAND_CONNECTIONS,
                                          mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=4),
                                          mp_drawing.DrawingSpec(color=(121, 44, 250), thickness=2, circle_radius=2)
                                          )

        cv2.imshow('Hand Tracking', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [33]:
##test##
import os
import mediapipe as mp
import cv2
import numpy as np
import uuid
import math
import time
import serial

ser = serial.Serial('COM6', 250000)
time.sleep(2)  # 시리얼 통신 안정화 대기

mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

# 거리 → 0~180 매핑 함수
def map_distance(d, min_d=0.15, max_d=0.4):
    d = np.clip(d, min_d, max_d)
    return int(180 * (1 - (d - min_d) / (max_d - min_d)))

def map_distance_t(d, min_d=0.22, max_d=0.32):
    d = np.clip(d, min_d, max_d)
    return int(180 * (1 - (d - min_d) / (max_d - min_d)))

# 각도 계산 함수 (세 점 A-B-C 기준 ∠ABC)
def calculate_adjusted_angle_with_direction(p0, p1, p2):
    v1 = np.array([p0.x - p1.x, p0.y - p1.y])  # 기준 벡터: p1 → p0
    v2 = np.array([p2.x - p1.x, p2.y - p1.y])  # 손가락 벡터: p1 → p2

    dot = np.dot(v1, v2)
    norm = np.linalg.norm(v1) * np.linalg.norm(v2)
    angle_rad = np.arccos(np.clip(dot / norm, -1.0, 1.0))
    angle_deg = np.degrees(angle_rad)
    diff = 180 - angle_deg

    # 방향 판단: 외적 z값 이용
    cross = v1[0] * v2[1] - v1[1] * v2[0]  # 2D 외적 (z 성분)
    if cross > 0:
        # 왼쪽으로 꺾임
        return int(90 - diff)
    else:
        # 오른쪽으로 꺾임
        return int(90 + diff)
        
def calculate_angle(a, b, c):
    v1 = np.array([a.x - b.x, a.y - b.y])
    v2 = np.array([c.x - b.x, c.y - b.y])
    dot = np.dot(v1, v2)
    norm = np.linalg.norm(v1) * np.linalg.norm(v2)
    angle = np.arccos(np.clip(dot / norm, -1.0, 1.0))
    return np.degrees(angle)

def calculate_angle(a, b, c):
    v1 = np.array([a.x - b.x, a.y - b.y])
    v2 = np.array([c.x - b.x, c.y - b.y])
    dot = np.dot(v1, v2)
    norm = np.linalg.norm(v1) * np.linalg.norm(v2)
    angle = np.arccos(np.clip(dot / norm, -1.0, 1.0))
    deg = np.degrees(angle)
    mapped = np.clip((deg / 40) * 60 + 120, 120, 180)
    return int(mapped)

# 관심 joint 인덱스
key_joints = [0,1,2,3,4,5,9,13,17,8,12,16,20]

# 연결선: 빠짐없이 이어지는 구조
draw_lines = [
    (0, 1), (1, 2), (2, 3), (3, 4),     # 엄지
    (0, 5), (5, 8),                    # 검지
    (0, 9), (9, 12),                   # 중지
    (0, 13), (13, 16),                # 약지
    (0, 17), (17, 20)                 # 소지
]

cap = cv2.VideoCapture(0)
last_sent_time = time.time()
with mp_hands.Hands(min_detection_confidence=0.8, min_tracking_confidence=0.5) as hands:
    while cap.isOpened():
        ret, frame = cap.read()
        frame = cv2.flip(frame, 1)
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = hands.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        current_time = time.time()

        h, w, _ = image.shape

        if results.multi_hand_landmarks:
            for hand in results.multi_hand_landmarks:
                lm = hand.landmark

                # joint만 그리기
                for i in key_joints:
                    cx, cy = int(lm[i].x * w), int(lm[i].y * h)
                    cv2.circle(image, (cx, cy), 6, (255, 255, 0), -1)

                # 연결선 그리기
                for start, end in draw_lines:
                    x1, y1 = int(lm[start].x * w), int(lm[start].y * h)
                    x2, y2 = int(lm[end].x * w), int(lm[end].y * h)
                    cv2.line(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

                # 거리 측정 및 매핑
                def dist(a, b):
                    return math.sqrt((lm[a].x - lm[b].x)**2 + (lm[a].y - lm[b].y)**2)
                
                dist5 = dist(0,4) # 엄지
                dist4 = dist(0,8)  # 검지
                dist3 = dist(0,12)  # 중지
                dist2 = dist(0,16)  # 약지
                dist1 = dist(0,20)  # 소지
                
                pairs = [dist1, dist2, dist3, dist4, dist5]
                for i, d in enumerate(pairs):
                    if i == 4:
                        mapped = map_distance_t(d)
                        cv2.putText(image, f"dist {i+1}:{mapped}", (10, 30 + i*25),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
                    else:
                        mapped = map_distance(d)
                        cv2.putText(image, f"dist {i+1}:{mapped}", (10, 30 + i*25),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

                # 각도 측정
                angle5 = calculate_angle(lm[2], lm[0], lm[5])
                angle4 = calculate_adjusted_angle_with_direction(lm[0], lm[5], lm[6])    # 검지
                angle3 = calculate_adjusted_angle_with_direction(lm[0], lm[9], lm[10])   # 중지
                angle2 = calculate_adjusted_angle_with_direction(lm[0], lm[13], lm[14])  # 약지
                angle1 = calculate_adjusted_angle_with_direction(lm[0], lm[17], lm[18])  # 소지

                angles = [angle1, angle2, angle3, angle4, angle5]
                for i, ang in enumerate(angles):
                    cv2.putText(image, f"Angle {i+1}: {ang}", (250, 30 + i*25),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 255), 2)
                
                 #Serial 통신으로 데이터 전송
                
                if current_time - last_sent_time >= 0.2:
                    finger_ids = [5, 4, 3, 2, 1]
                    for i in range(5):
                        fingerNum = finger_ids[i]
                        joint = map_distance(pairs[i])
                        angle = angles[i]
                        msg = f"{fingerNum},{joint},{angle}\n"
                        ser.write(msg.encode('utf-8'))
                    last_sent_time = current_time
                
                
        # 화면에 결과 출력
        cv2.imshow('Hand Analysis', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
        

cap.release()
cv2.destroyAllWindows()

In [3]:
##최종본##
import os
import mediapipe as mp
import cv2
import numpy as np
import uuid
import math
import time
import serial

ser = serial.Serial('COM6', 250000)
time.sleep(2)  # 시리얼 통신 안정화 대기

mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

# 거리 → 0~180 매핑 함수
def map_distance(d, min_d=0.15, max_d=0.4):
    d = np.clip(d, min_d, max_d)
    return int(180 * (1 - (d - min_d) / (max_d - min_d)))

# 각도 계산 함수 (세 점 A-B-C 기준 ∠ABC)
def calculate_adjusted_angle_with_direction(p0, p1, p2):
    v1 = np.array([p0.x - p1.x, p0.y - p1.y])  # 기준 벡터: p1 → p0
    v2 = np.array([p2.x - p1.x, p2.y - p1.y])  # 손가락 벡터: p1 → p2

    dot = np.dot(v1, v2)
    norm = np.linalg.norm(v1) * np.linalg.norm(v2)
    angle_rad = np.arccos(np.clip(dot / norm, -1.0, 1.0))
    angle_deg = np.degrees(angle_rad)
    diff = 180 - angle_deg

    # 방향 판단: 외적 z값 이용
    cross = v1[0] * v2[1] - v1[1] * v2[0]  # 2D 외적 (z 성분)
    if cross > 0:
        # 왼쪽으로 꺾임
        return int(90 - diff)
    else:
        # 오른쪽으로 꺾임
        return int(90 + diff)

# 관심 joint 인덱스
key_joints = [0,1,2,3,4,5,9,13,17,8,12,16,20]

# 연결선: 빠짐없이 이어지는 구조
draw_lines = [
    (0, 1), (1, 2), (2, 3), (3, 4),     # 엄지
    (0, 5), (5, 8),                    # 검지
    (0, 9), (9, 12),                   # 중지
    (0, 13), (13, 16),                # 약지
    (0, 17), (17, 20)                 # 소지
]

cap = cv2.VideoCapture(0)
last_sent_time = time.time()
with mp_hands.Hands(min_detection_confidence=0.8, min_tracking_confidence=0.5) as hands:
    while cap.isOpened():
        ret, frame = cap.read()
        frame = cv2.flip(frame, 1)
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = hands.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        current_time = time.time()

        h, w, _ = image.shape

        if results.multi_hand_landmarks:
            for hand in results.multi_hand_landmarks:
                lm = hand.landmark

                # joint만 그리기
                for i in key_joints:
                    cx, cy = int(lm[i].x * w), int(lm[i].y * h)
                    cv2.circle(image, (cx, cy), 6, (255, 255, 0), -1)

                # 연결선 그리기
                for start, end in draw_lines:
                    x1, y1 = int(lm[start].x * w), int(lm[start].y * h)
                    x2, y2 = int(lm[end].x * w), int(lm[end].y * h)
                    cv2.line(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

                # 거리 측정 및 매핑
                def dist(a, b):
                    return math.sqrt((lm[a].x - lm[b].x)**2 + (lm[a].y - lm[b].y)**2)

                dist4 = dist(0,8)  # 검지
                dist3 = dist(0,12)  # 중지
                dist2 = dist(0,16)  # 약지
                dist1 = dist(0,20)  # 소지
                pairs = [dist1, dist2, dist3, dist4]
                for i, d in enumerate(pairs):
                    mapped = map_distance(d)
                    cv2.putText(image, f"dist {i+1}:{mapped}", (10, 30 + i*25),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

                # 각도 측정
                angle4 = calculate_adjusted_angle_with_direction(lm[0], lm[5], lm[6])    # 검지
                angle3 = calculate_adjusted_angle_with_direction(lm[0], lm[9], lm[10])   # 중지
                angle2 = calculate_adjusted_angle_with_direction(lm[0], lm[13], lm[14])  # 약지
                angle1 = calculate_adjusted_angle_with_direction(lm[0], lm[17], lm[18])  # 소지

                angles = [angle1, angle2, angle3, angle4]
                for i, ang in enumerate(angles):
                    cv2.putText(image, f"Angle {i+1}: {ang}", (250, 30 + i*25),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 255), 2)
                
                 #Serial 통신으로 데이터 전송
                if current_time - last_sent_time >= 0.2:
                    finger_ids = [5, 4, 3, 2]
                    for i in range(4):
                        fingerNum = finger_ids[i]
                        joint = map_distance(pairs[i])
                        angle = angles[i]
                        msg = f"{fingerNum},{joint},{angle}\n"
                        ser.write(msg.encode('utf-8'))
                    last_sent_time = current_time
                
        # 화면에 결과 출력
        cv2.imshow('Hand Analysis', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
        

cap.release()
cv2.destroyAllWindows()